# Phase 2 - GitHub Tool Explore

GitHub REST API로 LangChain 관련 저장소를 검색하고 `search_github_repos` 도구로 래핑합니다.

완료 기준:
- 검색 결과에 레포 이름, URL, stars 포함
- API 실패 시 raise 대신 문자열 반환
- 사용자 질문 1회당 `github_search` API 호출 최대 2회 제한

## 1. 환경 로드

In [17]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "practice":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

github_token = os.getenv("GITHUB_TOKEN")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("GITHUB_TOKEN loaded:", bool(github_token))

PROJECT_ROOT: c:\Users\USER\Desktop\AI\langchain\PJ\langchain
GITHUB_TOKEN loaded: True


## 2. GitHub API 직접 테스트

In [18]:
import requests

from my_project.api_limits import create_query_limiter

query_limiter = create_query_limiter()
GITHUB_SEARCH_URL = "https://api.github.com/search/repositories"


def github_headers() -> dict[str, str]:
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        return {}
    return {"Authorization": f"Bearer {token}"}


def github_search_request(query: str, per_page: int = 10) -> requests.Response:
    params = {"q": f"langchain {query} in:name,description,readme", "sort": "stars", "per_page": per_page}
    return query_limiter.run(
        "github_search",
        lambda: requests.get(
            GITHUB_SEARCH_URL,
            headers=github_headers(),
            params=params,
            timeout=10,
        ),
    )


query_limiter.reset()
resp = github_search_request("rag", per_page=10)
print("status:", resp.status_code)
print("rate remaining:", resp.headers.get("X-RateLimit-Remaining"))
print("keys:", list(resp.json().keys())[:10])

status: 200
rate remaining: 29
keys: ['total_count', 'incomplete_results', 'items']


## 3. 응답 파싱 확인

In [19]:
items = resp.json().get("items", [])

for item in items[:5]:
    print(f"- {item['full_name']} (stars: {item['stargazers_count']})")
    print(f"  description: {item.get('description') or 'N/A'}")
    print(f"  url: {item['html_url']}")
    print()

- ollama/ollama (stars: 170962)
  description: Get up and running with Kimi-K2.5, GLM-5, MiniMax, DeepSeek, gpt-oss, Qwen, Gemma and other models.
  url: https://github.com/ollama/ollama

- microsoft/generative-ai-for-beginners (stars: 110355)
  description: 21 Lessons, Get Started Building with Generative AI 
  url: https://github.com/microsoft/generative-ai-for-beginners

- microsoft/Web-Dev-For-Beginners (stars: 95731)
  description: 24 Lessons, 12 Weeks, Get Started as a Web Developer
  url: https://github.com/microsoft/Web-Dev-For-Beginners

- punkpeye/awesome-mcp-servers (stars: 86452)
  description: A collection of MCP servers.
  url: https://github.com/punkpeye/awesome-mcp-servers

- mlabonne/llm-course (stars: 79075)
  description: Course to get into Large Language Models (LLMs) with roadmaps and Colab notebooks.
  url: https://github.com/mlabonne/llm-course



## 4. Tool 래핑

In [ ]:
from langchain_core.tools import tool
import re
import sys


# To see debug ADR, run this one line in a separate cell:
# os.environ["GITHUB_TOOL_DEBUG"] = "1"
ECOSYSTEM_KEYWORDS = {
    "rag": 2,
    "retrieval": 2,
    "agent": 2,
    "llm": 1,
    "chatbot": 1,
    "vector": 1,
    "embedding": 1,
    "multimodal": 1,
    "orchestration": 1,
    "document": 1,
    "qa": 1,
}
CORE_KEYWORDS = ("langchain", "langgraph")


def repo_search_text(item: dict) -> str:
    return " ".join([
        item.get("full_name") or "",
        item.get("name") or "",
        item.get("description") or "",
        " ".join(item.get("topics") or []),
    ]).lower()


def extract_query_terms(query: str) -> list[str]:
    """Use only ASCII tokens that can match GitHub metadata."""
    return [term.lower() for term in re.findall(r"[A-Za-z0-9]+", query) if len(term) >= 2]


def repo_relevance_score(item: dict, query: str) -> tuple[int, list[str]]:
    """Score only repos that pass the LangChain/LangGraph core gate."""
    text = repo_search_text(item)
    core_matches = [keyword for keyword in CORE_KEYWORDS if keyword in text]
    if not core_matches:
        return 0, []

    matched = [f"core:{keyword}" for keyword in core_matches]
    score = 5

    for term in extract_query_terms(query):
        if term in text:
            score += 3
            matched.append(f"query:{term}")

    for keyword, weight in ECOSYSTEM_KEYWORDS.items():
        if keyword in text:
            score += weight
            matched.append(f"ecosystem:{keyword}")

    stars = int(item.get("stargazers_count") or 0)
    stars_bonus = min(stars // 1000, 5)
    score += stars_bonus
    if stars_bonus:
        matched.append(f"stars:+{stars_bonus}")
    return score, matched


def select_relevant_repos(
    items: list[dict],
    query: str,
    limit: int = 3,
    min_score: int = 6,
) -> tuple[list[tuple[dict, int, list[str]]], bool]:
    scored = []
    gate_candidates = []
    for item in items:
        score, matched = repo_relevance_score(item, query)
        if score <= 0:
            continue
        row = (item, score, matched)
        gate_candidates.append(row)
        if score >= min_score:
            scored.append(row)

    sort_key = lambda row: (row[1], int(row[0].get("stargazers_count") or 0))
    if scored:
        return sorted(scored, key=sort_key, reverse=True)[:limit], False

    fallback = sorted(
        gate_candidates,
        key=lambda row: int(row[0].get("stargazers_count") or 0),
        reverse=True,
    )[:limit]
    return fallback, bool(fallback)


def relevance_label(score: int) -> str:
    if score >= 15:
        return "높음"
    if score >= 9:
        return "중간"
    return "낮음"


def build_github_debug_adr(
    query: str,
    response: requests.Response,
    candidate_count: int,
    selected_count: int,
    fallback_used: bool,
) -> str:
    """Observable debug record. It avoids exposing hidden chain-of-thought."""
    api_calls = query_limiter.snapshot().get("github_search", 0)
    return "\n".join([
        "\n[Debug / ADR]",
        "ADR-Phase2-001: GitHub repository search strategy",
        "- Context: Need LangChain implementation references similar to the user idea",
        f"- Decision: Use GitHub Search API with query 'langchain {query} in:name,description,readme'",
        "- Rationale: Fetch 10 candidates, then re-rank by LangChain gate, user query matches, ecosystem keywords, and stars bonus",
        f"- API: github_search call {api_calls}/2",
        f"- Status: HTTP {response.status_code}, rate_limit_remaining={response.headers.get('X-RateLimit-Remaining')}",
        f"- Candidate count: {candidate_count}",
        f"- Selected count: {selected_count}",
        f"- Fallback used: {fallback_used}",
        f"- Request URL: {response.url}",
    ])


def maybe_print_github_debug_adr(debug_adr: str) -> None:
    if os.getenv("GITHUB_TOOL_DEBUG") == "1":
        print(debug_adr, file=sys.stderr)


@tool
def search_github_repos(query: str) -> str:
    """Search GitHub repositories for LangChain implementation references. Pass English domain keywords only; do not include the word langchain."""
    try:
        response = github_search_request(query, per_page=10)
        response.raise_for_status()
        items = response.json().get("items", [])
        selected, fallback_used = select_relevant_repos(items, query, limit=3)
        debug_adr = build_github_debug_adr(query, response, len(items), len(selected), fallback_used)
        maybe_print_github_debug_adr(debug_adr)
        if not selected:
            return "관련 LangChain GitHub 레포지토리를 찾을 수 없습니다. 영어 키워드로 다시 검색해보세요."

        results = []
        if fallback_used:
            results.append("관련도 임계값을 넘는 레포가 없어 LangChain 게이트 통과 후보 중 stars 상위 결과를 반환합니다.")
        elif len(selected) < 3:
            results.append("GitHub 후보 10개 중 LangChain 게이트를 통과한 결과가 3개 미만이라 통과한 레포만 반환합니다.")

        for item, score, matched in selected:
            results.append(
                f"- {item['full_name']} (stars: {item['stargazers_count']})\n"
                f"  설명: {item.get('description') or 'N/A'}\n"
                f"  URL: {item['html_url']}\n"
                f"  관련도: {relevance_label(score)} (score: {score})\n"
                f"  근거 키워드: {', '.join(matched) or 'N/A'}"
            )
        return "\n\n".join(results)
    except Exception as exc:
        return f"GitHub 검색 실패: {exc}"


def ask_github(query: str) -> str:
    """사용자 질문 1회 처리. limiter를 리셋해서 질문당 2회 제한 적용."""
    query_limiter.reset()
    return search_github_repos.invoke(query)


# Test
for test_query in ["rag chatbot", "agent skill", "recipe recommendation"]:
    print(f"\n=== {test_query} ===")
    print(ask_github(test_query))
